# Author a portable experiment

This offline example uses the installed Workflows public API. No weather, database or engine is needed. WWR is dimensionless; categorical ventilation choices are labels. The seed makes independent samples reproducible. Physical models and weather must be explicitly supplied before execution.

In [ ]:
from epsm_workflows import WorkflowSpec
from epsm_workflows.distributions import normal, categorical

workflow = WorkflowSpec.template("monte_carlo", samples=12, seed=7)
workflow.vary("facade.wwr", min=0.1, max=0.4,
              distribution=normal(mean=0.25, std=0.05), baseline=0.2)
workflow.vary("systems.ventilation.topology", choices=["F", "FTX"],
              distribution=categorical(probabilities=[0.5, 0.5]))
rows = workflow.samples()
assert len(rows) == 12
assert all(0.1 <= row["facade.wwr"] <= 0.4 for row in rows)
assert rows == workflow.samples()
workflow

The persisted workflow and generated schema use the same typed definitions. This draft remains incomplete until subjects and weather are supplied; validation reports those requirements rather than selecting hidden assumptions.

In [ ]:
import json

restored = WorkflowSpec.from_dict(json.loads(json.dumps(workflow.to_dict())))
assert restored.id == workflow.id
assert "facade.wwr" in WorkflowSpec.schema()["x-epsm-inputs"]
requirements = restored.validate()
assert not requirements["ready"]
assert {item["code"] for item in requirements["diagnostics"]} >= {"missing_subjects", "missing_weather"}
requirements